In [1]:
import os
import numpy as np
import scipy.stats as stats
import torch
import copy
from torch import nn
from torch import optim
import torch.nn.functional as F
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [2]:
with_cuda = torch.cuda.is_available()
if with_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Translating sentences from English to French

The goal is to perform sequence-to-sequence prediction to translate English sentences to French.

We will use a simple dataset, made of two files: "small_vocab_fr.txt" and "small_vocab_en.txt".

**Question 1**

Take a look at the files. Then, build the class `CorpusWords`:
* two instances of this class will be created: one containing the French text, and the other the English text;
* each sentence will be augmented with a `<start>` token at the beginning, a `<end>` token at the end, and several `<empty>` tokens to make all sentences (in one language) have the same length;
* each corpus will be represented as two tensors:
  * `data`: a `LongTensor` of size (nb_sentences, max_sentence_length), containing the encoded sentences;
  * `lst_len`: a `LongTensor` of size (nb_sentences), containing the lengths of the sentences.

In [ ]:
class Dictionary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []
        self.nb_occ = {}

    def add_word(self, word):
        if word not in self.word2idx:
            self.idx2word.append(word)
            self.word2idx[word] = len(self.idx2word) - 1
            self.nb_occ[word] = 1
        else:
            self.nb_occ[word] += 1
        return self.word2idx[word]

    def __len__(self):
        return len(self.idx2word)


class CorpusWords:
    def __init__(self, path):
        self.dictionary = Dictionary()
        # Add special tokens first so they have fixed indices
        self.dictionary.add_word('<empty>')
        self.dictionary.add_word('<start>')
        self.dictionary.add_word('<end>')
        self.data, self.lst_len = self.tokenize(self.dictionary, path)

    def tokenize(self, dct, path):
        """Read file, build vocabulary, return (data tensor, lengths tensor)."""
        # --- First pass: read all sentences and build vocabulary ---
        sentences = []
        max_len = 0  # will include <start> and <end>
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                tokens = line.strip().split()
                if len(tokens) == 0:
                    continue
                for token in tokens:
                    dct.add_word(token)
                # +2 for <start> and <end>
                sentence_len = len(tokens) + 2
                if sentence_len > max_len:
                    max_len = sentence_len
                sentences.append(tokens)

        nb_sentences = len(sentences)

        # --- Second pass: encode each sentence ---
        start_idx = dct.word2idx['<start>']
        end_idx   = dct.word2idx['<end>']
        empty_idx = dct.word2idx['<empty>']
        #empty n'est pas nécessairement de token 0, et il faut ajouter des 0 à la fin de chaque phrase que l'on garde dans un tableau. 

        data    = torch.zeros(nb_sentences, max_len, dtype=torch.long)
        lst_len = torch.zeros(nb_sentences, dtype=torch.long)

        for i, tokens in enumerate(sentences):
            # actual length including <start> and <end>
            length = len(tokens) + 2
            lst_len[i] = length

            data[i, 0] = start_idx
            for j, token in enumerate(tokens):
                data[i, j + 1] = dct.word2idx[token]
            data[i, length - 1] = end_idx
            # remaining positions stay 0 (<empty>)

        return data, lst_len

**Question 2**

Build the dataset, which will be a `TensorDataset`, generating batches containing:
* the input sequences;
* their lengths;
* the target sequences;
* their lengths.

In [11]:
corpus_en = CorpusWords('small_vocab_en.txt')
corpus_fr = CorpusWords('small_vocab_fr.txt')

print(f"English vocabulary size : {len(corpus_en.dictionary)}")
print(f"French  vocabulary size : {len(corpus_fr.dictionary)}")
print(f"Number of sentences     : {corpus_en.data.shape[0]}")
print(f"Max EN sentence length  : {corpus_en.data.shape[1]}")
print(f"Max FR sentence length  : {corpus_fr.data.shape[1]}")

English vocabulary size : 230
French  vocabulary size : 358
Number of sentences     : 137860
Max EN sentence length  : 19
Max FR sentence length  : 25


In [12]:
from torch.utils.data import TensorDataset, DataLoader

# TensorDataset: (input_seq, input_len, target_seq, target_len)
dataset = TensorDataset(
    corpus_en.data,
    corpus_en.lst_len,
    corpus_fr.data,
    corpus_fr.lst_len
)

batch_size = 64
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Number of batches: {len(loader)}")

# Quick sanity-check
x, x_len, y, y_len = next(iter(loader))
print(f"Batch shapes — x: {x.shape}, x_len: {x_len.shape}, y: {y.shape}, y_len: {y_len.shape}")

Number of batches: 2155
Batch shapes — x: torch.Size([64, 19]), x_len: torch.Size([64]), y: torch.Size([64, 25]), y_len: torch.Size([64])


**Question 3**

Build a class `AttentionLayer`, performing self-attention.

In [17]:
class AttentionLayer(nn.Module):
    def __init__(self, n_feats: int) -> None:
        super().__init__()
        self.w = nn.Linear(
            in_features=n_feats,
            out_features=n_feats
        )
    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        w = self.w(X)
        output = F.softmax(torch.mul(X, w), dim=1)
        return output

Build a class `ModelAttention`, which performs sequence-to-sequence prediction with:
1. an embedding;
2. a LSTM (encoder);
3. an attention layer;
4. a LSTM (decoder);
5. a final layer.

Note: for performance, one can use `nn.utils.rnn.pack_padded_sequence` and `nn.utils.rnn.pad_packed_sequence` before and after the use of the LSTM encoder.

In [ ]:
class ModelAttention(nn.Module):
    def __init__(self, ntokens_enc, ntokens_dec, nembed, nhidden, nlayers):
        super(ModelAttention, self).__init__()
        self.nhidden = nhidden
        self.nlayers = nlayers
        self.embedding = nn.Embedding(ntokens_enc, nembed, padding_idx=0)

        self.encoder = nn.LSTM(
            input_size=nembed,
            hidden_size=nhidden,
            num_layers=nlayers,
            batch_first=True
        )
        self.attention = AttentionLayer(nhidden)
        self.decoder = nn.LSTM(
            input_size=nhidden,
            hidden_size=nhidden,
            num_layers=nlayers,
            batch_first=True
        )
        self.fc_out    = nn.Linear(nhidden, ntokens_dec)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, x, x_len):
        embedded = self.embedding(x)          
        x_len_cpu = x_len.cpu()                
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, x_len_cpu, batch_first=True, enforce_sorted=False
        )
        enc_out_packed, (h_n, c_n) = self.encoder(packed)
        enc_out, _ = nn.utils.rnn.pad_packed_sequence(
            enc_out_packed, batch_first=True
        )                                      
      
        context, _ = self.attention(enc_out)  

   
        dec_out, _ = self.decoder(context, (h_n, c_n)) 
      
        output = self.log_softmax(self.fc_out(dec_out))  
        return output

**Question 4**

Write the code to train the model.

In [ ]:
def train_model(loader, model, criterion, optimizer, nepochs):
    pass

**Question 5**

Perform the training and check the results.

In [ ]:
ntokens_enc = len(corpus_en.dictionary)
ntokens_dec = len(corpus_fr.dictionary)
nembed = 100
nhidden = 100
nlayers = 2

model = ModelAttention(ntokens_enc, ntokens_dec, nembed, nhidden, nlayers).to(device = device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

NameError: name 'corpus_en' is not defined

In [ ]:
criterion = nn.NLLLoss()

# optimizer
optimizer = optim.Adam(model.parameters(), lr=.001)

nepochs = 10

train_model(loader, model, criterion, optimizer, nepochs)